
# Spectral Bandwidth Extraction

## Aim
This notebook extracts **spectral bandwidth** from preprocessed beehive recordings and aligns the feature with environmental measurements such as **temperature** and **humidity**.

Spectral bandwidth describes how widely acoustic energy is distributed around the spectral centroid. In the context of hive acoustics, it can help reveal changes in the complexity or spread of bee-generated sound over time.



## Notes
This refined version keeps the same overall workflow as the original notebook, but restructures it into a cleaner and more reusable pipeline for dissertation work and GitHub portfolio use.

Because the preprocessing stage normalised the recordings, the extracted feature should be interpreted primarily as a measure of **relative spectral change over time**, rather than absolute acoustic intensity.


In [ ]:
import os
from glob import glob
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
from sklearn.preprocessing import MinMaxScaler

## Configuration

In [ ]:
# Audio feature settings
TARGET_SR = 16000
N_FFT = 2048
HOP_SAMPLES = 16000     # 1 feature value per second at 16 kHz
CHUNK_SECONDS = 3600    # 1-hour visualisation chunks

# Session-specific paths
INPUT_FOLDER = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024"
OUTPUT_FOLDER = os.path.join(INPUT_FOLDER, "SB_features_refined")

ENV_PATH = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024\Env_logs\HaniBi_20240815T134229+0200.xlsx"

BASE_START_TIME = pd.Timestamp("2024-08-13 11:20:00")
START_SLICE = pd.Timestamp("2024-08-13 11:20:00")
END_SLICE = pd.Timestamp("2024-08-15 13:04:41")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


## Why spectral bandwidth?
Spectral bandwidth measures the spread of frequencies around the centroid.

In beehive recordings:

- **Lower bandwidth** may indicate a more stable, tonal, and consistent buzzing pattern.
- **Higher bandwidth** may indicate broader spectral activity, potentially reflecting increased movement, agitation, environmental disturbance, or more complex acoustic behaviour.

This makes it a useful complement to RMS, MFCCs, spectral centroid, and rolloff.


## Helper functions

In [ ]:
def extract_spectral_bandwidth(audio_path, start_time, n_fft=N_FFT, hop_length=HOP_SAMPLES):
    """
    Extract spectral bandwidth from one audio file and attach absolute timestamps.
    """
    y, sr = librosa.load(audio_path, sr=None)

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length
    ).squeeze()

    times = librosa.frames_to_time(
        np.arange(len(bandwidth)),
        sr=sr,
        hop_length=hop_length
    )

    timestamps = [start_time + timedelta(seconds=float(t)) for t in times]

    return pd.DataFrame({
        "timestamp": timestamps,
        "spectral_bandwidth": bandwidth
    })


def load_and_interpolate_environment(env_path, start_slice, end_slice):
    """
    Load environmental log data, restrict it to the recording interval,
    and interpolate to 1-second resolution for alignment with extracted features.
    """
    env_df = pd.read_excel(env_path)
    env_df = env_df.rename(columns={"Date": "timestamp"})
    env_df["timestamp"] = pd.to_datetime(env_df["timestamp"])

    env_df = env_df[
        (env_df["timestamp"] >= start_slice) &
        (env_df["timestamp"] <= end_slice)
    ]

    env_df = env_df[["timestamp", "Temperatura (°C)", "Wilgotność (%)"]]
    env_df = (
        env_df.set_index("timestamp")
        .resample("1S")
        .interpolate(method="linear")
        .reset_index()
    )

    return env_df


def build_file_start_times(file_list, base_start_time):
    """
    Estimate the real start timestamp of each consecutive WAV file
    using cumulative file durations.
    """
    durations = [librosa.get_duration(path=path) for path in file_list]

    start_times = []
    elapsed = 0.0
    for duration in durations:
        start_times.append(base_start_time + timedelta(seconds=float(elapsed)))
        elapsed += duration

    return start_times


def process_session(input_folder, output_folder, env_df, base_start_time):
    """
    Extract spectral bandwidth for each WAV file, align with environmental data,
    and save one CSV per file.
    """
    file_list = sorted(glob(os.path.join(input_folder, "*.wav")))
    start_times = build_file_start_times(file_list, base_start_time)

    processing_log = []

    for file_path, file_start_time in zip(file_list, start_times):
        filename = os.path.basename(file_path)
        print(f"Processing: {filename}")

        try:
            feature_df = extract_spectral_bandwidth(file_path, file_start_time)

            merged_df = pd.merge_asof(
                feature_df.sort_values("timestamp"),
                env_df.sort_values("timestamp"),
                on="timestamp",
                direction="nearest"
            )

            output_csv = os.path.join(
                output_folder,
                f"{os.path.splitext(filename)[0]}_spectral_bandwidth.csv"
            )
            merged_df.to_csv(output_csv, index=False)

            processing_log.append({
                "file": filename,
                "status": "success",
                "rows_saved": len(merged_df),
                "output_csv": output_csv
            })

        except Exception as e:
            processing_log.append({
                "file": filename,
                "status": "failed",
                "rows_saved": 0,
                "output_csv": "",
                "error": str(e)
            })
            print(f"Error processing {filename}: {e}")

    return pd.DataFrame(processing_log)

## Load and prepare environmental data

In [ ]:
env_df = load_and_interpolate_environment(
    env_path=ENV_PATH,
    start_slice=START_SLICE,
    end_slice=END_SLICE
)

env_df.head()

## Batch extraction and alignment

In [ ]:
processing_log = process_session(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    env_df=env_df,
    base_start_time=BASE_START_TIME
)

processing_log

In [ ]:
processing_log.to_csv(
    os.path.join(OUTPUT_FOLDER, "spectral_bandwidth_processing_log.csv"),
    index=False
)

## Combine exported feature files

In [ ]:
sb_files = sorted(glob(os.path.join(OUTPUT_FOLDER, "*_spectral_bandwidth.csv")))

combined_df = pd.concat(
    [pd.read_csv(f, parse_dates=["timestamp"]) for f in sb_files],
    ignore_index=True
)

combined_df = combined_df.dropna(
    subset=["spectral_bandwidth", "Temperatura (°C)", "Wilgotność (%)"]
)

combined_df.head()


## Normalisation for comparative visualisation
Min-max normalisation is used here only to place spectral bandwidth, temperature, and humidity on the same plotting scale.

This does **not** replace the original feature values and should be treated as a visual comparison tool rather than an analytical transformation for interpretation of absolute magnitude.


In [ ]:
scaler = MinMaxScaler()

norm_values = scaler.fit_transform(
    combined_df[["spectral_bandwidth", "Temperatura (°C)", "Wilgotność (%)"]]
)

norm_df = pd.DataFrame(
    norm_values,
    columns=["sb_norm", "temp_norm", "hum_norm"]
)

norm_df["timestamp"] = combined_df["timestamp"].values
norm_df["elapsed_sec"] = (
    norm_df["timestamp"] - norm_df["timestamp"].iloc[0]
).dt.total_seconds()

norm_df.head()

## Plotting in time chunks

In [ ]:
max_time = norm_df["elapsed_sec"].max()
num_chunks = int(np.ceil(max_time / CHUNK_SECONDS))

for chunk_idx in range(num_chunks):
    start_sec = chunk_idx * CHUNK_SECONDS
    end_sec = (chunk_idx + 1) * CHUNK_SECONDS

    chunk = norm_df[
        (norm_df["elapsed_sec"] >= start_sec) &
        (norm_df["elapsed_sec"] < end_sec)
    ]

    if chunk.empty:
        continue

    plt.figure(figsize=(14, 5))
    plt.plot(chunk["timestamp"], chunk["sb_norm"], label="Spectral Bandwidth (norm)")
    plt.plot(chunk["timestamp"], chunk["temp_norm"], label="Temperature (norm)")
    plt.plot(chunk["timestamp"], chunk["hum_norm"], label="Humidity (norm)")

    plt.title(f"Spectral Bandwidth vs Temperature and Humidity | Chunk {chunk_idx + 1}")
    plt.xlabel("Timestamp")
    plt.ylabel("Normalised Value")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Optional summary statistics

In [ ]:
combined_df[["spectral_bandwidth", "Temperatura (°C)", "Wilgotność (%)"]].describe()


## Interpretation
Spectral bandwidth reflects the spread of frequencies present in the hive soundscape at each analysis frame.

Potential interpretations in this project include:

- **lower spectral bandwidth**: more concentrated or stable acoustic activity, possibly linked to calmer collective buzzing
- **higher spectral bandwidth**: broader frequency distribution, potentially associated with increased movement, disturbance, or more complex colony behaviour
- **temporal variation**: day-night changes or event-related shifts may reveal transitions in hive state
- **environmental alignment**: comparing bandwidth with temperature and humidity may help identify whether environmental conditions co-vary with changes in colony acoustics

Because the recordings were preprocessed and normalised earlier in the pipeline, these results are best interpreted as **relative spectral behaviour over time**.
